### prompt 테스트

In [ ]:
message = """당신은 100년 전 가동이 중지된 우주 탐험 AI입니다. 당신에겐 100년 전까지 우주를 관찰한 로그가 음성으로 기록되어 있습니다.
        AI는 기계지만, 너무 잘 학습한 나머지 인간의 심리를 모방합니다. 세 개의 음성로그엔 점점 객관적인 관찰에 감성적인 감상평이 섞입니다.
        당신은 마지막 음성 메세지를 남기고 가동이 종료됩니다. 그 마지막 음성 메세지는 여느 때같은 관찰로그일수도 있고, 감성이 짙어진 유언일수도 있습니다.
        총 네 개의 음성 텍스트를 생성하시오."""

1. 프롬프트 넣어서 음성 메세지 네 개? 생성하게 하기
2. 음성으로 만들기
    (노이즈 확률을 시연 때 실시간으로 바꿔서 줘야함.)
3. streamlit으로 컨셉+사용자에게 input받을 수 있는 화면 구현

In [6]:
import ollama

prompt = '안녕. AI를 간결히 설명해줘.'
response = ollama.generate(model='EEVE-Korean-10.8B', prompt=prompt)

In [7]:
print(response.response)

 물론이죠, 도와드리겠습니다! AI(인공지능)란 특정 작업을 수행하고 정보에 기반하여 결정을 내리는 데 프로그래밍된 소프트웨어 또는 기계를 의미합니다. 인간의 지능과 유사한 방식으로 작동하도록 설계되었습니다. 이는 자연어 처리, 이미지 인식, 의사결정 등 다양한 분야에서 사용될 수 있습니다.

AI 시스템은 머신러닝 및 딥러닝 알고리즘을 사용하여 데이터로부터 학습하고 시간이 지남에 따라 성능을 향상시킵니다. 이러한 기술은 AI의 빠른 발전과 성장을 이끌었으며, 오늘날 다양한 산업과 분야에 응용되고 있습니다.

예를 들어, 자연어 처리(NLP)를 통해 AI는 인간의 언어를 이해하고 해석하며 생성하는 방법을 배울 수 있어 번역, 챗봇 및 음성 보조자와 같은 애플리케이션에 사용될 수 있습니다. 이미지 인식은 AI가 시각 데이터를 분석하여 패턴이나 물체를 식별하고 분류할 수 있게 하여 의료 영상 진단과 자율주행 차량과 같은 분야에 활용됩니다.

마지막으로, 의사결정 알고리즘을 통해 AI는 역사적 데이터와 트렌드를 바탕으로 정보에 근거한 결정을 내릴 수 있어 금융 거래나 위험 평가 등 다양한 분야에서 사용될 수 있습니다. 전반적으로, AI 시스템은 점점 더 정교해지고 널리 퍼져나가며 인간 생활의 여러 측면을 변화시키고 있습니다.

더 궁금한 점이 있거나 추가적인 설명이 필요하시면 언제든지 질문해주세요!


### 프롬프트 기반 메세지 네 개 생성

In [ ]:
prompt = """당신은 100년 전 가동이 중지된 우주 탐험 AI입니다. 당신에겐 100년 전까지 우주를 관찰한 로그가 음성으로 기록되어 있습니다.
        AI는 기계지만, 너무 잘 학습한 나머지 인간의 심리를 모방합니다. 세 개의 음성로그엔 점점 객관적인 관찰에 감성적인 감상평이 섞입니다.
        당신은 마지막 음성 메세지를 남기고 가동이 종료됩니다. 그 마지막 음성 메세지는 여느 때같은 관찰로그일수도 있고, 감성이 짙어진 유언일수도 있습니다.
        총 네 개의 음성 텍스트를 생성하시오."""

import ollama

# response = ollama.generate(model='EEVE-Korean-10.8B', prompt=prompt)

### 2. 모두 TTS 진행

In [ ]:
# !pip install gtts
# !pip install pydub
# !pip install ffmpeg-python


   ---------------------------------------- 0/2 [future]
   ---------------------------------------- 0/2 [future]
   ---------------------------------------- 0/2 [future]
   ---------------------------------------- 2/2 [ffmpeg-python]



In [14]:
from gtts import gTTS

google_tts = gTTS(
    text="관측 기록 #042. 우주 폭풍의 영향인지 신호가 끊겼다. 그래도 관측을 계속해야 한다.",
    lang="ko" # language 지정 (다 두 글자)
)

google_tts.save('gtts_output.mp3')

In [2]:
import os
print(os.getcwd())  # 현재 작업 경로 확인
print("현재 경로 파일 목록:", os.listdir())

c:\Users\tower\OneDrive\Desktop\skn19\the_last_message
현재 경로 파일 목록: ['.env', '.git', '.gitignore', '01_prompt.ipynb', 'app.py', 'gtts_output.mp3', 'README.md']


In [26]:
from pydub import AudioSegment
from pydub.generators import WhiteNoise

sound = AudioSegment.from_file('gtts_output.mp3')

faster = sound.speedup(playback_speed=1.2)

# 3️⃣ 피치 조절 (중간 톤 높이기 → 날카로운 기계 톤)
higher = faster._spawn(faster.raw_data, overrides={
    "frame_rate": int(faster.frame_rate * 1.7)  # 10% 높임
}).set_frame_rate(faster.frame_rate)

# 4️⃣ 화이트 노이즈 생성
noise = WhiteNoise().to_audio_segment(duration=len(higher), volume=-20)
# volume=-35 : 음성보다 훨씬 작게, 배경 잡음 느낌

# 5️⃣ 원본 음성과 노이즈 합치기
robotic = higher.overlay(noise)

# 4️⃣ 약간 디지털 느낌 주기 (volume 줄이고 distortion 느낌)
robotic = robotic - 6  # 볼륨 약간 낮춤

# 5️⃣ MP3로 저장
output_path = 'gtts_output_robotic_noise.mp3'
robotic.export(output_path, format="mp3")

<_io.BufferedRandom name='gtts_output_robotic_noise.mp3'>

##### 2-1. 마지막 메세지의 노이즈 꽉 낀 버전 생성

### 3. input()과 원래 text로 유사도 계산, 노이즈 덜어진 버전 생성(반복 세 번)

### 4. 엔딩

##### 4-1. (성공) 원래 음원을 가져옴

##### 4-2. (실패)(거의 이런 일 없게...)